#### SET UP

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
from pathlib import Path
from ucimlrepo import fetch_ucirepo

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Fetch UCI dataset
diabetes = fetch_ucirepo(id=296)

# Load the original dataset
df = diabetes.data.original.copy()

print("Loaded UCI dataset.")
print("Shape:", df.shape)

### BASIC PROFILING
Shape, dtypes, missingness, and a first look at the native `readmitted` field before we touch anything.

In [ ]:
print("Rows, columns:", df.shape)
print("\nUnique patients (patient_nbr):", df["patient_nbr"].nunique())
print("Rows per patient (top 5):")
print(df["patient_nbr"].value_counts().head())

Construct the target variable
Here we look at the distribution of the data, check for multiple readmissions and sector the data into 

```
Y = 1  if readmitted == "<30"
Y = 0  otherwise (">30" or "NO")
```

In [ ]:
#check how skewed the data is 
encounter_counts = df["patient_nbr"].value_counts()
print(encounter_counts.describe())
print("\nPatients with only 1 encounter:", (encounter_counts == 1).sum())
print("Patients with 2+ encounters:", (encounter_counts >= 2).sum())
print("Patients with 5+ encounters:", (encounter_counts >= 5).sum())

In [ ]:
# DECISION LOG (spec §1.1): keep every encounter as its own row.
# Patient-level leakage will be prevented at split-time via GroupShuffleSplit on patient_nbr, not by deduplicating rows here.

df["readmitted"].value_counts()

In [ ]:
#building binary target variable Y, where 1 = readmitted within 30 days, 0 = not readmitted or readmitted after 30 days

df["Y"] = (df["readmitted"] == "<30").astype(int)
df["Y"].value_counts()

### EXCLUSIONS DUE TO DEATH

In [ ]:
# Codes confirmed against IDs_mapping.csv (downloaded from UCI, cross-checked against multiple independent analyses of this dataset).
DEATH_CODES = [11, 19, 20, 21]
HOSPICE_CODES = [13, 14]

n_before = len(df)
excluded_death = df["discharge_disposition_id"].isin(DEATH_CODES).sum()
excluded_hospice = df["discharge_disposition_id"].isin(HOSPICE_CODES).sum()

print(f"Rows flagged as death at discharge: {excluded_death}")
print(f"Rows flagged as hospice discharge: {excluded_hospice}")

In [ ]:
##apply exclusion criteria to remove rows flagged as death or hospice discharge

df_audited = df[~df["discharge_disposition_id"].isin(DEATH_CODES + HOSPICE_CODES)].copy()

n_after = len(df_audited)
print(f"Cohort size before exclusions: {n_before}")
print(f"Cohort size after exclusions:  {n_after}")
print(f"Removed: {n_before - n_after} rows ({(n_before - n_after) / n_before:.2%})")

In [ ]:
df_audited["Y"] = (df_audited["readmitted"] == "<30").astype(int)
print(df_audited["Y"].value_counts(normalize=True))

### LEAKAGE AUDIT

In [ ]:
leakage_audit = pd.DataFrame({"column": df_audited.columns})

post_discharge_cols = ["readmitted"]
identifier_cols = ["encounter_id", "patient_nbr"]
review_cols = ["payer_code"]
sparse_cols = ["weight"]
constant_cols = ["citoglipton", "examide"]

def classify(col):
    if col in post_discharge_cols:
        return "post_discharge"
    if col in identifier_cols:
        return "identifier"
    if col in review_cols:
        return "needs_review"
    if col in sparse_cols:
        return "excessively_sparse"
    if col in constant_cols:
        return "constant"
    return "known_at_discharge"

leakage_audit["leakage_status"] = leakage_audit["column"].apply(classify)
leakage_audit["leakage_status"].value_counts()

### SAVE DATA

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_audited.to_csv(PROCESSED_DIR / "audited_cohort.csv", index=False)
leakage_audit.to_csv(PROCESSED_DIR / "leakage_audit.csv", index=False)

print("Saved:")
print(f" - {PROCESSED_DIR / 'audited_cohort.csv'}  (shape: {df_audited.shape})")
print(f" - {PROCESSED_DIR / 'leakage_audit.csv'}   (shape: {leakage_audit.shape})")

### SUMMARY

What this notebook did and why it matters

Before building any predictive model, we ran a data audit on the raw dataset. This is to make sure the model would learn genuine clinical risk patterns, not artifacts of how the data happens to be structured. Three things came out of it:

We defined the target precisely. "Readmitted within 30 days" sounds simple, but the raw data actually distinguishes three outcomes (no readmission, readmission after 30 days, readmission within 30 days). We made an explicit, documented choice about which counts as a "positive" case, rather than letting that ambiguity leak into modeling decisions later.

We removed a class of patients who couldn't validly be scored. Patients who died during their hospital stay, or were discharged to hospice, cannot be "readmitted" by definition. Keeping them in would have taught the model to associate certain patient profiles with low risk, when in fact it was measuring something else entirely (mortality, not readmission risk). This is the kind of error that's invisible in a model's accuracy score but would seriously undermine its validity.


We audited every column for two separate risks: information leakage and usability. Leakage means a piece of information wasn't actually available at the moment a real clinician would need the prediction — using it would make the model look artificially accurate in testing while being useless in practice. Usability means some columns were so sparse (one field was 97% missing) or so uniform (two medications were never prescribed to anyone in this dataset) that they couldn't meaningfully inform the model regardless of leakage concerns. We also caught and fixed a subtle data-loading bug, where two lab-result columns had "test not performed" silently miscoded as "missing data" — a distinction that matters clinically and would have caused us to needlessly discard two real, usable features.